<a href="https://colab.research.google.com/github/mc66000/fairwashing_aduit_AI/blob/main/fairwashing_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys, sklearn, scipy, pandas, matplotlib
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print("pandas:", pandas.__version__)
print("matplotlib:", matplotlib.__version__)

Python: 3.13.18
scikit-learn: 1.6.1
scipy: 1.16.3
pandas: 2.2.3
matplotlib: 3.10.0


In [3]:
from sklearn.datasets import fetch_openml
X, y = fetch_openml("adult", version=2, as_frame=True, return_X_y=True)
print("資料大小:", X.shape)
print(X.head())
print(y.value_counts())

資料大小: (48842, 14)
   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country

In [4]:
df = X.copy()
df["income"] = y
df.to_csv("adult.csv", index=False)

from google.colab import files
files.download("adult.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
!pip freeze > requirements.txt
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

D 部份：用小實驗理解四個統計概念（約 20 分鐘）
光看定義不容易懂，我們直接用模擬來「看到」它們。

概念 1：Demographic parity gap
假設男性 100 人中錄取 30 人，女性 100 人中錄取 20 人：

男性錄取率 = 0.30，女性錄取率 = 0.20
DP gap = 0.30 − 0.20 = 0.10
gap 越接近 0，代表兩性的錄取率越接近。平台用替身模型，就是要讓審計 API 顯示的 gap 接近 0。

概念 2–4：誤報、漏報、power
情境：獨立證據的雜訊率是 10%，所以平台誠實時，API 和獨立紀錄大約有 10% 的答案不一致。稽核者抽查 100 筆，如果不一致的比例「明顯超過 10%」，就判定平台造假。

新開一格執行：

In [6]:
import numpy as np
from scipy.stats import binomtest

rng = np.random.default_rng(42)
n, eps = 100, 0.10          # 抽查 100 筆，已知雜訊 10%

def one_audit(true_rate):
    k = rng.binomial(n, true_rate)                 # 模擬：實際有幾筆不一致
    p = binomtest(k, n, eps, alternative="greater").pvalue
    return p < 0.05                                # True = 判定造假

honest = np.mean([one_audit(0.10) for _ in range(1000)])  # 平台誠實
cheat  = np.mean([one_audit(0.20) for _ in range(1000)])  # 平台造假，不一致率升到 20%

print(f"平台誠實時，被判造假的比例（誤報率 / Type I error）: {honest:.3f}")
print(f"平台造假時，被抓到的比例（power）: {cheat:.3f}")
print(f"平台造假時，沒被抓到的比例（漏報率 / Type II error）: {1 - cheat:.3f}")

平台誠實時，被判造假的比例（誤報率 / Type I error）: 0.036
平台造假時，被抓到的比例（power）: 0.887
平台造假時，沒被抓到的比例（漏報率 / Type II error）: 0.113


In [9]:
import numpy as np
from scipy.stats import binomtest

rng = np.random.default_rng(42)
n, eps = 30, 0.10         # 抽查 30 筆，已知雜訊 10%

def one_audit(true_rate):
    k = rng.binomial(n, true_rate)                 # 模擬：實際有幾筆不一致
    p = binomtest(k, n, eps, alternative="greater").pvalue
    return p < 0.05                                # True = 判定造假

honest = np.mean([one_audit(0.10) for _ in range(1000)])  # 平台誠實
cheat  = np.mean([one_audit(0.20) for _ in range(1000)])  # 平台造假，不一致率升到 20%

print(f"平台誠實時，被判造假的比例（誤報率 / Type I error）: {honest:.3f}")
print(f"平台造假時，被抓到的比例（power）: {cheat:.3f}")
print(f"平台造假時，沒被抓到的比例（漏報率 / Type II error）: {1 - cheat:.3f}")

平台誠實時，被判造假的比例（誤報率 / Type I error）: 0.027
平台造假時，被抓到的比例（power）: 0.402
平台造假時，沒被抓到的比例（漏報率 / Type II error）: 0.598
